In [14]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from pydantic import BaseModel,Field
from typing import TypedDict,Literal
from dotenv import load_dotenv

In [15]:
load_dotenv()
model = ChatOpenAI(model="gpt-4o-mini")

In [16]:
class SentimentSchema(BaseModel):
    sentiment:Literal['positive','negative']=Field(description="Sentiment of the review")

In [17]:
structure_output = model.with_structured_output(SentimentSchema)

In [18]:
class ReviewState(TypedDict):
    review:str
    sentiment:Literal['positive','negative']
    diagnosis:dict
    response:str

In [19]:
def find_sentiment(state:ReviewState):
    prompt = f" for the follwoing review find the sentiment \n {state['review']}"
    sentiment=structure_output.invoke(prompt)
    return{'sentiment':sentiment}

In [20]:
def check_sentiment(state:ReviewState):
    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

In [21]:
def positive_response(state:ReviewState):
    prompt = f"""
    Write a warm thankyou message in resposne to this review \n\n\"{state['review']}\" \n
    Also,kindly ask the user to leave feedback on  our website
    """
    result = model.invoke(prompt).content
    return{
        "response":result
    }

In [22]:
class DiagnosisSchema(BaseModel):
    issue_type:Literal['UX','Support','Bug','Other']=Field(description="The category of the issue mentioned in the review")
    tone:Literal['angry','frustrated','disappointed','clam']=Field(description="The emotional tone expressed by the user")
    urgency:Literal['low','medium','high']=Field(description="How urgent or critical the issue appears to be")

In [23]:
structure_output2 = model.with_structured_output(DiagnosisSchema)

In [24]:
def run_diagnosis(state:ReviewState):
    prompt = f"""
    Diagnosis this negative review"\n\n{state['review']}\n"
    "Return issue_type,tone,urgency
    """
    response = structure_output2.invoke(prompt)
    return {
        "diagnosis":response.model_dump()
    }

In [25]:
def negative_response(state:ReviewState):
    diagnosis=state['diagnosis']
    
    prompt =f"""
    You are a support assistant.
    The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency '{diagnosis['urgency']}'
    Write an empathetic, helpful resolution message
    """
    
    response = model.invoke(prompt).content
    return{
        'response':response
    }

In [26]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment',find_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('negative_response',negative_response)
graph.add_node('run_diagnosis',run_diagnosis)

graph.add_edge(START,'find_sentiment')

graph.add_conditional_edges('find_sentiment',check_sentiment)

graph.add_edge('positive_response',END)
graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('negative_response',END)

workflow = graph.compile()

In [ ]:
initial_state={
    "review":"The product was bad"
}

result = workflow.invoke(initial_state)
print(result)

{'review': 'The product was bad', 'sentiment': SentimentSchema(sentiment='negative'), 'diagnosis': {'issue_type': 'Other', 'tone': 'disappointed', 'urgency': 'medium'}, 'response': "Subject: We're Here to Help\n\nDear [User's Name],\n\nThank you for reaching out to us. I’m sorry to hear that you’re feeling disappointed. It’s important to us that your experience is a positive one, and I understand how frustrating it can be when things don’t go as expected.\n\nTo assist you better, could you please provide a bit more detail about the specific issue you’re facing? This will help us address your concerns more effectively and find the right solution for you.\n\nWe appreciate your patience, and we’re here to help you every step of the way. If there’s anything specific you would like to know or discuss further, please don’t hesitate to share.\n\nLooking forward to your response.\n\nBest regards,\n\n[Your Name]  \n[Your Position]  \n[Your Contact Information]  "}
